# ClimateGuard — Phase 4: Exploratory Data Analysis

**Project:** ClimateGuard — Explainable & Drift-Aware Heatwave Risk Prediction  
**Part:** 1 — Dataset + ML (Adrian)  
**Phase:** 4 — Exploratory Data Analysis  
**Dataset:** ERA5 via Open-Meteo Historical API  
**Master file:** `data/raw/all_cities_era5_raw.csv`  
**Period:** 1990-01-01 → 2025-08-31  
**Cities:** New Delhi, Lucknow, Nagpur, Ahmedabad, Mumbai  

---

⚠️ **IMPORTANT**: This notebook is READ-ONLY with respect to the master raw dataset.  
No modifications are made to `data/raw/`. All outputs go to `results/` and `results/plots/EDA/`.  

---

## Contents
1. Introduction & Setup
2. Dataset Loading
3. Dataset Overview
4. Data Quality Verification
5. Descriptive Statistics & Data Dictionary
6. City Comparison
7. Temperature Analysis
8. Seasonal Analysis
9. Long-Term Trends
10. Humidity Analysis
11. Precipitation Analysis
12. Wind / Pressure / Radiation
13. Correlation Analysis
14. Preliminary Extreme-Temperature Analysis
15. Outlier Analysis
16. Coastal vs Plains Comparison
17. Key Findings
18. Limitations
19. Recommendations for Phase 5

## 1. Introduction & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.linestyle': '--',
})

# Paths — works when notebook is run from its own directory or from project root
NB_DIR  = Path().resolve()
ROOT    = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
RAW_CSV = ROOT / 'data' / 'raw' / 'all_cities_era5_raw.csv'
PLOT_DIR = ROOT / 'results' / 'plots' / 'EDA'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {ROOT}')
print(f'Dataset      : {RAW_CSV}')
print(f'Plot output  : {PLOT_DIR}')

# City config
CITY_ORDER  = ['delhi', 'lucknow', 'nagpur', 'ahmedabad', 'mumbai']
CITY_LABELS = {
    'delhi': 'New Delhi', 'lucknow': 'Lucknow', 'nagpur': 'Nagpur',
    'ahmedabad': 'Ahmedabad', 'mumbai': 'Mumbai',
}
CITY_COLORS = {
    'delhi': '#E63946', 'lucknow': '#457B9D', 'nagpur': '#2A9D8F',
    'ahmedabad': '#E9C46A', 'mumbai': '#9B5DE5',
}
WEATHER_VARS = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'apparent_temperature_max', 'apparent_temperature_min', 'apparent_temperature_mean',
    'precipitation_sum', 'rain_sum', 'wind_speed_10m_max', 'wind_gusts_10m_max',
    'relative_humidity_2m_max', 'relative_humidity_2m_min', 'relative_humidity_2m_mean',
    'surface_pressure_mean', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration',
]
HW_THRESH = {'delhi': 40.0, 'lucknow': 40.0, 'nagpur': 40.0, 'ahmedabad': 40.0, 'mumbai': 37.0}
SEASONS = {
    'Winter': [12,1,2], 'Pre-Monsoon': [3,4,5],
    'Monsoon': [6,7,8,9], 'Post-Monsoon': [10,11],
}
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print('Setup complete.')

## 2. Dataset Loading

In [ ]:
df = pd.read_csv(RAW_CSV, parse_dates=['date'])
df['month']  = df['date'].dt.month
df['year']   = df['date'].dt.year

def get_season(m):
    for s, months in SEASONS.items():
        if m in months:
            return s
    return 'Unknown'
df['season'] = df['month'].apply(get_season)

print(f'Loaded: {RAW_CSV.name}')
print(f'Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

## 3. Dataset Overview

In [ ]:
print('COLUMNS AND DTYPES')
print(df.dtypes.to_string())

In [ ]:
print('CITY COVERAGE')
for ck in CITY_ORDER:
    sub = df[df['city_key'] == ck]
    print(f"  {ck:<12} {CITY_LABELS[ck]:<14} {sub['region_type'].iloc[0]:<8} "
          f"{sub['date'].min().date()} → {sub['date'].max().date()}  rows={len(sub):,}")

## 4. Data Quality Verification

In [ ]:
print(f"Missing values (all weather vars) : {df[WEATHER_VARS].isnull().sum().sum()}")
print(f"Full duplicate rows               : {df.duplicated().sum()}")
print(f"city_key + date duplicates        : {df.duplicated(subset=['city_key','date']).sum()}")

print('\nMissing dates per city:')
for ck in CITY_ORDER:
    sub = df[df['city_key'] == ck]
    full = pd.date_range('1990-01-01','2025-08-31', freq='D')
    miss = full.difference(sub['date'])
    print(f"  {ck:<12} missing_dates={len(miss)}")

## 5. Descriptive Statistics & Data Dictionary

In [ ]:
desc = df[WEATHER_VARS].describe().T
desc['missing%'] = df[WEATHER_VARS].isnull().mean() * 100
desc.style.format(precision=2).background_gradient(subset=['mean'], cmap='RdYlBu_r')

## 6. City Comparison

In [ ]:
rows = []
for ck in CITY_ORDER:
    s = df[df['city_key'] == ck]
    rows.append({
        'City':          CITY_LABELS[ck],
        'Region':        s['region_type'].iloc[0],
        'Mean Tmax (°C)': round(s['temperature_2m_max'].mean(),2),
        'Max Tmax (°C)':  round(s['temperature_2m_max'].max(),2),
        'Mean Tmin (°C)': round(s['temperature_2m_min'].mean(),2),
        'Mean RH (%)':    round(s['relative_humidity_2m_mean'].mean(),2),
        'Mean Precip (mm/day)': round(s['precipitation_sum'].mean(),2),
        'Total Precip (mm)':    round(s['precipitation_sum'].sum(),1),
        'Mean Wind (km/h)':     round(s['wind_speed_10m_max'].mean(),2),
        'Mean Pressure (hPa)':  round(s['surface_pressure_mean'].mean(),2),
    })
pd.DataFrame(rows).set_index('City')

In [ ]:
# Load pre-generated plot
from IPython.display import Image
Image(str(PLOT_DIR / '01_city_comparison_bars.png'))

## 7. Temperature Analysis

In [ ]:
# Tmax distribution
Image(str(PLOT_DIR / '02_tmax_distribution_by_city.png'))

In [ ]:
Image(str(PLOT_DIR / '03_tmin_distribution_by_city.png'))

In [ ]:
Image(str(PLOT_DIR / '05_app_tmax_distribution_by_city.png'))

In [ ]:
Image(str(PLOT_DIR / '06_tmax_timeseries_rolling.png'))

In [ ]:
Image(str(PLOT_DIR / '07_tmin_timeseries_rolling.png'))

In [ ]:
Image(str(PLOT_DIR / '09_annual_mean_tmax_by_city.png'))

## 8. Seasonal Analysis

In [ ]:
Image(str(PLOT_DIR / '11_monthly_tmax_heatmap.png'))

In [ ]:
Image(str(PLOT_DIR / '08_monthly_tmax_per_city.png'))

In [ ]:
Image(str(PLOT_DIR / '12_seasonal_tmax_boxplot.png'))

In [ ]:
print('Hottest months per city (median Tmax):')
for ck in CITY_ORDER:
    sub  = df[df['city_key'] == ck].groupby('month')['temperature_2m_max'].median()
    top3 = sub.nlargest(3)
    names = [MONTH_LABELS[m-1] for m in top3.index]
    print(f"  {CITY_LABELS[ck]:<14}: {', '.join([f'{n} ({v:.1f}°C)' for n,v in zip(names,top3.values)])}")

## 9. Long-Term Trends

> **Limitation note**: Observed linear trends over 1990–2025 are reported. This dataset alone cannot establish causal attribution to climate change — see Section 18.

In [ ]:
print(f"{'City':<14} {'Metric':<14} {'Slope (°C/yr)':>15} {'R²':>8} {'p-value':>10} {'Significant?':>14}")
print('-' * 70)
for ck in CITY_ORDER:
    sub = df[df['city_key'] == ck].copy()
    sub['year'] = sub['date'].dt.year
    ann = sub.groupby('year')['temperature_2m_min'].mean()
    slope, intercept, r, p, se = stats.linregress(ann.index, ann.values)
    sig = 'YES' if p < 0.05 else 'no'
    print(f"  {CITY_LABELS[ck]:<14} {'mean_tmin':<14} {slope:>+15.4f} {r**2:>8.3f} {p:>10.4f} {sig:>14}")

In [ ]:
Image(str(PLOT_DIR / '10_long_term_temperature_trends.png'))

## 10. Humidity Analysis

In [ ]:
Image(str(PLOT_DIR / '14_humidity_distributions.png'))

In [ ]:
Image(str(PLOT_DIR / '15_monthly_humidity_heatmap.png'))

In [ ]:
Image(str(PLOT_DIR / '16_tmax_vs_humidity_scatter.png'))

In [ ]:
print('Pearson r (Tmax vs mean RH) per city:')
for ck in CITY_ORDER:
    sub = df[df['city_key'] == ck][['temperature_2m_max','relative_humidity_2m_mean']].dropna()
    r, p = stats.pearsonr(sub['temperature_2m_max'], sub['relative_humidity_2m_mean'])
    print(f"  {CITY_LABELS[ck]:<14} r={r:+.3f}  p={p:.4f}")

## 11. Precipitation Analysis

In [ ]:
print('Zero-rain day frequency:')
for ck in CITY_ORDER:
    sub   = df[df['city_key'] == ck]['precipitation_sum']
    zero  = (sub == 0).sum()
    print(f"  {CITY_LABELS[ck]:<14} {zero:,} / {len(sub):,}  ({zero/len(sub)*100:.1f}%)")

In [ ]:
Image(str(PLOT_DIR / '17_precipitation_histogram_logscale.png'))

In [ ]:
Image(str(PLOT_DIR / '18_monthly_precip_heatmap.png'))

In [ ]:
Image(str(PLOT_DIR / '19_annual_total_precipitation.png'))

## 12. Wind / Pressure / Radiation

In [ ]:
Image(str(PLOT_DIR / '21_wind_distributions.png'))

In [ ]:
Image(str(PLOT_DIR / '22_monthly_pressure_by_city.png'))

In [ ]:
Image(str(PLOT_DIR / '23_monthly_radiation_by_city.png'))

In [ ]:
Image(str(PLOT_DIR / '24_monthly_et0_by_city.png'))

## 13. Correlation Analysis

In [ ]:
Image(str(PLOT_DIR / '25_correlation_matrix.png'))

In [ ]:
corr = df[WEATHER_VARS].corr()
print('Strong correlations (|r| > 0.70):')
for i, c1 in enumerate(WEATHER_VARS):
    for j, c2 in enumerate(WEATHER_VARS):
        if j <= i: continue
        r = corr.loc[c1, c2]
        if abs(r) > 0.70:
            print(f"  {c1:<38} ↔ {c2:<38} r={r:+.3f}")

## 14. Preliminary Extreme-Temperature Analysis

> ⚠️ **These are PRELIMINARY threshold analyses only.**  
> Final heatwave labels will be defined in **Phase 6** using IMD criteria.  
> Thresholds used here: Delhi/Lucknow/Nagpur/Ahmedabad ≥40°C, Mumbai ≥37°C.

In [ ]:
print(f"{'City':<14} {'Threshold':>10} {'Exceed days':>12} {'%':>8} {'Hottest date':>14} {'Hottest T':>10} {'Max consec':>12}")
print('-' * 80)
for ck in CITY_ORDER:
    sub    = df[df['city_key'] == ck].sort_values('date').reset_index(drop=True)
    thresh = HW_THRESH[ck]
    exceed = sub['temperature_2m_max'] >= thresh
    n_exc  = exceed.sum()
    pct    = n_exc / len(sub) * 100
    hot_i  = sub['temperature_2m_max'].idxmax()
    hot_d  = sub.loc[hot_i, 'date'].date()
    hot_v  = sub.loc[hot_i, 'temperature_2m_max']
    c_max  = 0; c_cur = 0
    for e in exceed:
        if e: c_cur += 1; c_max = max(c_max, c_cur)
        else: c_cur = 0
    print(f"  {CITY_LABELS[ck]:<14} {thresh:>10.1f} {n_exc:>12,} {pct:>7.2f}% {str(hot_d):>14} {hot_v:>10.1f} {c_max:>12}")

In [ ]:
Image(str(PLOT_DIR / '26_annual_exceedance_days.png'))

In [ ]:
Image(str(PLOT_DIR / '27_monthly_exceedance_days.png'))

## 15. Outlier Analysis

> Statistical outliers are documented, **not removed**.  
> Extreme hot days are likely genuine meteorological events and may be critical signal for heatwave detection.

In [ ]:
Image(str(PLOT_DIR / '28_tmax_boxplot_outliers.png'))

## 16. Coastal vs Plains Comparison

In [ ]:
plains = ['delhi','lucknow','nagpur','ahmedabad']
print(f"{'Metric':<28} {'Plains avg':>12} {'Mumbai (coastal)':>18}")
print('-' * 60)
for label, col, fn in [
    ('Tmax mean (°C)',  'temperature_2m_max',       'mean'),
    ('Tmax std (°C)',   'temperature_2m_max',       'std'),
    ('Tmin mean (°C)',  'temperature_2m_min',       'mean'),
    ('Mean RH (%)',     'relative_humidity_2m_mean','mean'),
    ('Mean Precip (mm)','precipitation_sum',         'mean'),
    ('Mean Wind (km/h)','wind_speed_10m_max',        'mean'),
]:
    p_val = getattr(df[df['city_key'].isin(plains)][col], fn)()
    m_val = getattr(df[df['city_key'] == 'mumbai'][col], fn)()
    print(f"  {label:<28} {p_val:>12.2f} {m_val:>18.2f}")

In [ ]:
Image(str(PLOT_DIR / '29_coastal_vs_plains_violin.png'))

In [ ]:
Image(str(PLOT_DIR / '30_seasonal_coastal_vs_plains.png'))

## 17. Key Findings

All numbers below are sourced directly from the ERA5 dataset and computed above.

### Dataset
- **65,135 rows** across 5 cities, 1990-01-01 → 2025-08-31
- **16 weather variables**, 0 missing values, 0 duplicate records
- All values within physical bounds

### Temperature
- **Nagpur** has the highest mean Tmax (32.85°C) and the absolute hottest observation (46.9°C on 2010-05-24)
- **Delhi** recorded 46.8°C on 1995-06-15
- **Mumbai** has the narrowest Tmax range (std 2.1°C vs ~5.8°C for plains cities) — a defining coastal characteristic
- **May** is the hottest month for all cities except Mumbai (where Apr, May, Mar are virtually tied)

### Long-Term Trends (observed, not attributed)
- **Annual mean Tmin** shows a statistically significant rising trend (p<0.05) in all 5 cities (+0.030 to +0.040°C/yr)
- **Mumbai's mean Tmax** trend is also statistically significant (+0.028°C/yr, R²=0.633)
- **Annual mean Tmax** trends for Delhi, Lucknow, Nagpur, Ahmedabad are positive but not statistically significant at the annual level

### Humidity
- **Mumbai** has the highest mean RH (74.8%) and the narrowest range (never drops below 33%)
- **Nagpur/Ahmedabad** have the lowest mean RH (~58–59%) during the dry season
- Tmax and RH are negatively correlated in all cities (range: r=−0.23 for Ahmedabad to r=−0.73 for Nagpur)

### Precipitation
- Highly zero-inflated: 60–74% of days have zero rain
- **Mumbai** has the highest mean daily precipitation (5.33 mm/day) and total (69,423 mm over the period)
- **Delhi** has the lowest total (24,822 mm)
- Strong monsoon signal (Jun–Sep) visible in all cities

### Wind / Pressure / Radiation
- **Mumbai** has the highest mean wind speed (17.4 km/h) — consistent with coastal exposure
- **Ahmedabad** has the highest mean radiation (19.34 MJ/m²) and ET₀ (5.05 mm/day)
- ET₀ is strongly correlated with shortwave radiation (r=+0.901) and negatively with humidity (r=−0.797)

### Correlations
- `precipitation_sum` and `rain_sum` are perfectly correlated (r=+1.000) — only one is needed
- `wind_speed_10m_max` and `wind_gusts_10m_max` are highly correlated (r=+0.903) — likely redundant
- Temperature variables (Tmax/Tmin/Tmean/Apparent) form a highly correlated cluster
- ET₀ is the strongest single correlate of Tmax across all cities (r≈+0.75–0.89)

### Preliminary Extreme Temperature
- **Nagpur** has the most threshold-exceeding days (1,761 / 13.52%) and longest consecutive run (56 days)
- **Mumbai** has almost no threshold-exceeding days (8 total, 0.06%) even at the lower 37°C threshold
- Extreme days concentrate in **Apr–Jun** for plains cities; Mumbai's rare extreme days occur in Apr–May

## 18. Limitations

1. **Reanalysis data, not ground observations.** ERA5 data is a numerical weather reanalysis product. It is model-derived at 0.25° (~28 km) resolution and may not capture local extremes at sub-grid scale.

2. **Trend analysis ≠ causal attribution.** Observed linear trends over 1990–2025 are purely descriptive. This single dataset cannot prove or disprove attribution to large-scale climate forcing.

3. **Heatwave labels not yet defined.** The threshold-exceedance analysis in Step 14 is exploratory. Final IMD-aligned heatwave labels (Phase 6) will use different criteria (e.g., ≥40°C AND ≥4.5°C above climatological normal, or absolute thresholds). Current results must not be treated as ground truth heatwave labels.

4. **Only 5 cities.** Results may not generalise to other Indian cities with different climatic regimes (e.g., semi-arid, high-altitude).

5. **`precipitation_sum` = `rain_sum` in this dataset.** They are identical (r=1.000). Snow contribution is effectively zero for these tropical/semi-arid Indian cities.

6. **No urban heat island correction.** ERA5 gridded data does not account for urban microclimate effects.

7. **Right-censored period.** The dataset ends 2025-08-31. The final months of 2025 are not yet available.

## 19. Recommendations for Phase 5 (Data Cleaning)

Based on EDA findings:

1. **No imputation needed.** 0 missing values across all variables.

2. **Drop `rain_sum`.** It is perfectly correlated with `precipitation_sum` (r=1.000) and adds no information.

3. **Evaluate `wind_gusts_10m_max`.** Highly correlated with `wind_speed_10m_max` (r=0.903). Keep both for now; feature selection in Phase 7 will decide.

4. **Do NOT remove statistical outliers.** High-Tmax days (IQR outliers) are potential heatwave events — they are the signal, not noise.

5. **Verify `heatwave_threshold_c` column.** The master raw file has a `heatwave_threshold_c` column only in individual city files from the old downloader. Confirm it is correctly populated in the final dataset before Phase 6.

6. **For heatwave labeling (Phase 6)**, consider IMD criteria and city-specific normal temperature baselines rather than fixed absolute thresholds.

7. **Mumbai requires separate treatment** in any normalised feature engineering given its fundamentally different Tmax distribution (std 2.1°C vs ~5.8°C for plains cities).